In [2]:
import json
with open("./vocabularies.json", "r") as f:
    wordlists = json.load(f)
total_vocabs = set()
for batch_key in wordlists:
    total_vocabs.update(
        sum(wordlists[batch_key].values(), [])
    )

In [1]:
import tqdm
import datrie
import os
detokenize = lambda token_indices, id_to_token: "".join([id_to_token[i] for i in token_indices])
def tokenize(current_expr, token_to_id, trie):
    token_indices = []
    while current_expr:
        match = trie.longest_prefix(current_expr)
        if match:
            token_indices.append(token_to_id[match])
            current_expr = current_expr[len(match):]
    return token_indices

In [15]:
def test_vocab_set_extensiveness(vocabularies):
    id_to_token = {i: vocab for i, vocab in enumerate(vocabularies)}
    token_to_id = {vocab: i for i, vocab in enumerate(vocabularies)}
    some_trie = datrie.Trie(sorted(list(set("".join(vocabularies)))))
    for token, _index in token_to_id.items():
        some_trie[token] = _index
    # some_trie.longest_prefix("-10")
    # return
    all_batch_ids = [f"0{i}" for i in range(10)] + list(range(10, 99))
    for batch_idx in tqdm.tqdm(range(len(all_batch_ids)), desc="Batch"):
        batch_id = all_batch_ids[batch_idx]
        all_files_batch_00 = os.listdir(f"../../cad-recode-v1.5/train/batch_{batch_id}")
        for i in tqdm.tqdm(range(len(all_files_batch_00)), desc=f"Batch {batch_id}"):
            file_name = all_files_batch_00[i]
            with open(f"../../cad-recode-v1.5/train/batch_{batch_id}/{file_name}", "r") as f:
                cad = f.readlines()
                cad_expression = "".join(cad[1:]).replace("\n", ";")
                try:
                    tokenized = tokenize(cad_expression, token_to_id, some_trie)
                    detokenized = detokenize(tokenized, id_to_token)
                    assert cad_expression == detokenized, f"Mismatch: {cad_expression} != {detokenized}"
                except:
                    raise Exception(f"Tokenization failed for {cad_expression}")

In [16]:
vocabularies = [
    "cq.Workplane(",
    ".arc(",
    ".assemble(",
    ".box(",
    ".circle(",
    ".close(",
    ".cylinder(",
    ".extrude(",
    ".face(",
    ".finalize(",
    ".moveTo(",
    ".push(",
    ".rect(",
    ".reset(",
    ".segment(",
    ".sketch(",
    ".union(",
    ".workplane(",
    "(", ")", "[", "]",
    "0", "1", "2", "3", "4", "5", "6", "7", "8", "9", ".",
    "-", ",", "/",  ";",
    "origin=", "mode=", "offset=",
    "r=", "w0=", "w1=", "w0", "w1",
    "'a'", "'s'", "'i'", "'c'", "'r'",
    "'XY'", "'YZ'", "'ZX'"
]

In [ ]:
test_vocab_set_extensiveness(vocabularies)

Batch:   8%|▊         | 8/99 [00:12<02:27,  1.62s/it]